# 57. The float32 experiment: `enable_x64(False)`

**Objectives:**

- Explain why `dalitzplotfitter` enables float64/complex128 by default on import (a deliberate
  deviation from JAX's own default).
- Opt back out with `enable_x64(False)` and rerun a small toy fit under float32.
- Observe (and honestly report, whichever way it goes) whatever precision degradation or
  convergence issue appears, rather than silently working around it.
- Restore float64 with `enable_x64(True)` at the end.

Run the cells in order in a fresh kernel. Masses are in GeV, invariants in GeV^2.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # The package default -- float64/complex128 end to end.

import jax
import numpy as np

from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, Parameter, RealImag, Resonance,
    FitSession, generate_toy,
)

## 1. Why float64 is the package default

Per `CLAUDE.md`: importing `dalitzplotfitter` enables JAX 64-bit precision automatically
(unless `JAX_ENABLE_X64` was already set in the environment, which stays authoritative) -- the
project deliberately runs float64/complex128 for amplitude-analysis stability, and this is
*not* JAX's own default (JAX defaults to float32 unless told otherwise).
`enable_x64(False)` remains available to opt back into float32 for an explicit, validated
experiment, but nothing in the numerical path is expected to be correct or to converge reliably
under float32 -- Minuit's EDM-based convergence check in particular. This notebook is that
experiment.

In [2]:
import jax.numpy as jnp

print("x64 enabled (default):", jax.config.jax_enable_x64)
print("default float dtype:", jnp.array(1.0).dtype)
print("default complex dtype:", jnp.array(1.0 + 0j).dtype)

x64 enabled (default): True
default float dtype: float64
default complex dtype: complex128


## 2. Build and fit a small toy model at float64 (baseline)

A minimal rho + non-resonant model on `D+ -> pi- pi+ pi+`, fit under the package default
precision, to have a baseline for comparison.

In [3]:
def coeff(name, x0, y0):
    return RealImag(
        Parameter.coefficient(f"{name}.x", x0, owner=name, bounds=(-3.0, 3.0)),
        Parameter.coefficient(f"{name}.y", y0, owner=name, bounds=(-3.0, 3.0)),
    )

def build_model():
    channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
    return DecayModel(
        channel,
        components=[
            Resonance("rho", (0, 1), coeff("rho", 1.0, 0.0), mass=0.7753, width=0.1491, spin=1),
            NonResonant(coeff("NR", 0.3, -0.1), name="NR"),
        ],
        normalization_method="square-dalitz", normalization_resolution=40,
    )

model64 = build_model()
truth = {p.name: p.value for p in model64.parameters}
data64 = generate_toy(
    model64, 800, parameters=truth, seed=57,
    method="inverse-transform", inverse_resolution=200, include_momenta=False,
)
start = {"rho.x": 0.9, "rho.y": 0.15, "NR.x": 0.25, "NR.y": -0.05}
session64 = FitSession(model64, data64)
result64 = session64.fit(start, simplex=True, ncall=4000)
session64.report(result64)
print(f"float64: valid={bool(result64.valid)}  edm={float(result64.fmin.edm):.3e}")

valid=True  NLL=702.218520
parameter                           value            error
rho.x                            0.999828          1.11168
rho.y                           0.0736101          1.53753
NR.x                             0.296919         0.343733
NR.y                           -0.0414405         0.478626


Fit fractions (physical)
component                    fraction [%]
rho                                91.792
NR                                  8.208
sum                               100.000
float64: valid=True  edm=2.164e-13


## 3. Rerun under float32

`enable_x64(False)` must be called before building new JAX arrays/models -- existing traced
computations from the float64 model above are unaffected, so a fresh model/data/session is
built here entirely under float32.

In [4]:
enable_x64(False)
print("x64 enabled:", jax.config.jax_enable_x64)

model32 = build_model()
data32 = generate_toy(
    model32, 800, parameters=truth, seed=57,
    method="inverse-transform", inverse_resolution=200, include_momenta=False,
)
print("s12 dtype:", data32.s12.dtype)

session32 = FitSession(model32, data32)
result32 = session32.fit(start, simplex=True, ncall=4000)
session32.report(result32)
print(f"float32: valid={bool(result32.valid)}  edm={float(result32.fmin.edm):.3e}")

x64 enabled: False


/home/juan-leite/Work/DalitzPlotFitter/src/dalitzplotfitter/toy_accept.py:23: UserWarning: Explicitly requested dtype float64 requested in ones is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  result = jnp.ones((size,), dtype=jnp.float64)


/home/juan-leite/Work/DalitzPlotFitter/src/dalitzplotfitter/amplitude/components.py:67: UserWarning: Explicitly requested dtype complex128 requested in full is not available, and will be truncated to dtype complex64. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return jnp.full((size,), self.value_, dtype=jnp.complex128)


s12 dtype: float32


/home/juan-leite/Work/DalitzPlotFitter/src/dalitzplotfitter/workflow.py:102: UserWarning: Explicitly requested dtype float64 requested in ones is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  values = jnp.ones((size,), dtype=jnp.float64)


valid=False  NLL=702.242310
parameter                           value            error
rho.x                             1.00007         0.157537
rho.y                           0.0753813          1.58186
NR.x                             0.297027         0.163191
NR.y                           -0.0409959         0.475632


/home/juan-leite/Work/DalitzPlotFitter/src/dalitzplotfitter/amplitude/components.py:67: UserWarning: Explicitly requested dtype complex128 requested in full is not available, and will be truncated to dtype complex64. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return jnp.full((size,), self.value_, dtype=jnp.complex128)


Fit fractions (physical)
component                    fraction [%]
rho                                91.795
NR                                  8.205
sum                               100.000
float32: valid=False  edm=1.613e-04


## 4. Compare

Values, errors and the EDM convergence diagnostic side by side. Per `CLAUDE.md`, a
degraded EDM/error estimate or an outright convergence failure under float32 is the expected
teaching point here -- not a bug to silently work around. If the fit happens to still land
close to the float64 result at this tiny toy scale, that is also a valid, honestly-reported
observation.

In [5]:
print(f"{'parameter':10s} {'float64':>12s} {'float32':>12s} {'|diff|':>12s}")
for name in truth:
    v64 = float(result64.values[name])
    v32 = float(result32.values[name])
    print(f"{name:10s} {v64:12.6f} {v32:12.6f} {abs(v64 - v32):12.2e}")

print()
print(f"float64: valid={bool(result64.valid)}  edm={float(result64.fmin.edm):.3e}")
print(f"float32: valid={bool(result32.valid)}  edm={float(result32.fmin.edm):.3e}")
if not result32.valid or float(result32.fmin.edm) > 10 * max(float(result64.fmin.edm), 1e-10):
    print("-> float32 shows the expected degradation (invalid fit and/or much worse EDM).")
else:
    print("-> at this toy's small scale, float32 happened to converge comparably; "
          "CLAUDE.md's warning is about reliability in general, not a guaranteed failure "
          "on every input.")

parameter       float64      float32       |diff|
rho.x          0.999828     1.000067     2.39e-04
rho.y          0.073610     0.075381     1.77e-03
NR.x           0.296919     0.297027     1.07e-04
NR.y          -0.041441    -0.040996     4.45e-04

float64: valid=True  edm=2.164e-13
float32: valid=False  edm=1.613e-04
-> float32 shows the expected degradation (invalid fit and/or much worse EDM).


## 5. Restore float64

Always restore the package default before any further work in a shared kernel/session.

In [6]:
enable_x64(True)
print("x64 enabled (restored):", jax.config.jax_enable_x64)

x64 enabled (restored): True


## Continue learning

See `CLAUDE.md` and `src/dalitzplotfitter/config.py`'s `enable_x64` docstring. Float32 is
useful only for explicit, validated experiments (e.g. exploring device/memory limits); it is
never a substitute for the package's default float64 numerical path in a real fit.

Return to [the course guide](TUTORIALS.md).